In [1]:
import os
import random
import pandas as pd
import numpy as np

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

from skopt import BayesSearchCV
from skopt.space import Categorical


# =====================================================
# Reproducibility
# =====================================================

RANDOM_STATE = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# =====================================================
# Settings
# =====================================================

CHANNEL = "C4"

# Path to the folder containing the feature CSV files
base_dir = "path_to_feature_files"

# Folder where all output files will be saved
results_dir = "results"

file_path = os.path.join(base_dir, f"Feat_{CHANNEL}_final.csv")
output_dir = os.path.join(results_dir, f"Bayesian_LOSO_RF_{CHANNEL}")

os.makedirs(output_dir, exist_ok=True)

N_ITER = 10
threshold_values = np.arange(0.1, 1.0, 0.1)


# =====================================================
# Load data
# =====================================================

df = pd.read_csv(file_path)
df = df.sort_values(by=["Subject_ID", "Segment"]).reset_index(drop=True)

meta_cols = ["Segment", "Subject_ID", "Subject", "Label"]

X = df.drop(columns=meta_cols, errors="ignore")
y = df["Label"]
groups = df["Subject_ID"]

feature_names = X.columns.tolist()


# =====================================================
# RF pipeline
# =====================================================

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE))
])


# =====================================================
# RF hyperparameter range from Table II
# Note: "auto" is removed because newer sklearn versions may not support it.
# =====================================================

search_space = {
    "clf__n_estimators": Categorical([10, 20, 30, 40, 50, 60, 70, 80, 90, 100]),
    "clf__max_depth": Categorical([3, 5, 7, 8]),
    "clf__min_samples_split": Categorical([10, 15, 20, 25, 50]),
    "clf__min_samples_leaf": Categorical([5, 10, 15, 20, 25]),
    "clf__bootstrap": Categorical([True, False]),
    "clf__max_features": Categorical(["sqrt", "log2", None]),
    "clf__max_leaf_nodes": Categorical([10, 15, 20, 25])
}


# =====================================================
# Storage
# =====================================================

subject_true = []
subject_pred = []
selected_thresholds = []

fold_summary_logs = []
bayes_search_logs = []
threshold_logs = []
inner_validation_logs = []
test_subject_logs = []
test_segment_prediction_logs = []
feature_rank_logs = []


# =====================================================
# Outer LOSO
# =====================================================

outer_logo = LeaveOneGroupOut()

for outer_fold, (train_idx, test_idx) in enumerate(
    outer_logo.split(X, y, groups), 1
):

    print("\n====================================")
    print(f"Outer Fold: {outer_fold}")
    print("====================================")

    X_train_outer = X.iloc[train_idx]
    y_train_outer = y.iloc[train_idx]
    groups_train_outer = groups.iloc[train_idx]

    X_test_outer = X.iloc[test_idx]
    y_test_outer = y.iloc[test_idx]
    groups_test_outer = groups.iloc[test_idx]

    test_subject = groups_test_outer.unique()[0]

    print(f"Channel: {CHANNEL}")
    print(f"Test Subject: {test_subject}")

    inner_logo = LeaveOneGroupOut()

    bayes_search = BayesSearchCV(
        estimator=pipe,
        search_spaces=search_space,
        n_iter=N_ITER,
        cv=inner_logo.split(X_train_outer, y_train_outer, groups_train_outer),
        scoring="accuracy",
        n_jobs=1,
        random_state=RANDOM_STATE,
        verbose=0,
        refit=True
    )

    bayes_search.fit(X_train_outer, y_train_outer)

    best_model = bayes_search.best_estimator_
    best_params = bayes_search.best_params_
    best_cv_score = bayes_search.best_score_

    print("Best Parameters:")
    print(best_params)
    print(f"Best Inner CV Accuracy: {best_cv_score:.4f}")

    cv_results_df = pd.DataFrame(bayes_search.cv_results_)
    cv_results_df["channel"] = CHANNEL
    cv_results_df["model"] = "RF"
    cv_results_df["outer_fold"] = outer_fold
    cv_results_df["test_subject"] = test_subject
    bayes_search_logs.append(cv_results_df)

    # =====================================================
    # Threshold selection
    # =====================================================

    threshold_scores = []

    for threshold in threshold_values:

        inner_true = []
        inner_pred = []

        for inner_fold, (inner_train_idx, inner_val_idx) in enumerate(
            inner_logo.split(X_train_outer, y_train_outer, groups_train_outer), 1
        ):

            X_inner_train = X_train_outer.iloc[inner_train_idx]
            y_inner_train = y_train_outer.iloc[inner_train_idx]

            X_inner_val = X_train_outer.iloc[inner_val_idx]
            y_inner_val = y_train_outer.iloc[inner_val_idx]
            groups_inner_val = groups_train_outer.iloc[inner_val_idx]

            val_subject = groups_inner_val.unique()[0]

            best_model.fit(X_inner_train, y_inner_train)
            val_segment_preds = best_model.predict(X_inner_val)

            mdd_ratio = np.mean(val_segment_preds == 1)

            val_subject_pred = 1 if mdd_ratio >= threshold else 0
            val_subject_true = y_inner_val.iloc[0]

            inner_true.append(val_subject_true)
            inner_pred.append(val_subject_pred)

            inner_validation_logs.append({
                "channel": CHANNEL,
                "model": "RF",
                "outer_fold": outer_fold,
                "test_subject": test_subject,
                "inner_fold": inner_fold,
                "validation_subject": val_subject,
                "threshold": threshold,
                "true_label": val_subject_true,
                "predicted_label": val_subject_pred,
                "mdd_segment_ratio": mdd_ratio,
                "num_segments": len(val_segment_preds),
                "num_predicted_mdd_segments": np.sum(val_segment_preds == 1),
                "num_predicted_control_segments": np.sum(val_segment_preds == 0)
            })

        threshold_acc = accuracy_score(inner_true, inner_pred)
        threshold_scores.append(threshold_acc)

        threshold_logs.append({
            "channel": CHANNEL,
            "model": "RF",
            "outer_fold": outer_fold,
            "test_subject": test_subject,
            "threshold": threshold,
            "threshold_accuracy": threshold_acc
        })

        print(f"Threshold {threshold:.1f} Accuracy: {threshold_acc:.4f}")

    best_threshold = threshold_values[np.argmax(threshold_scores)]
    best_threshold_score = np.max(threshold_scores)

    selected_thresholds.append(best_threshold)

    print(f"Selected Threshold: {best_threshold:.1f}")
    print(f"Best Threshold Accuracy: {best_threshold_score:.4f}")

    # =====================================================
    # Train final model
    # =====================================================

    best_model.fit(X_train_outer, y_train_outer)

    # =====================================================
    # RF feature ranking
    # =====================================================

    final_rf_model = best_model.named_steps["clf"]
    feature_importances = final_rf_model.feature_importances_

    feature_rank_df = pd.DataFrame({
        "channel": CHANNEL,
        "model": "RF",
        "outer_fold": outer_fold,
        "test_subject": test_subject,
        "feature": feature_names,
        "importance": feature_importances
    })

    feature_rank_df["rank"] = feature_rank_df["importance"].rank(
        ascending=False,
        method="dense"
    ).astype(int)

    feature_rank_df = feature_rank_df.sort_values(
        by=["rank", "importance"],
        ascending=[True, False]
    )

    feature_rank_logs.append(feature_rank_df)

    print("\nTop 10 ranked features for this fold:")
    print(feature_rank_df.head(10))

    # =====================================================
    # Testing
    # =====================================================

    test_segment_preds = best_model.predict(X_test_outer)
    test_mdd_ratio = np.mean(test_segment_preds == 1)

    final_subject_pred = 1 if test_mdd_ratio >= best_threshold else 0
    final_subject_true = y_test_outer.iloc[0]

    subject_true.append(final_subject_true)
    subject_pred.append(final_subject_pred)

    num_mdd_segments = np.sum(test_segment_preds == 1)
    num_control_segments = np.sum(test_segment_preds == 0)
    num_segments = len(test_segment_preds)

    print(f"Test MDD Segment Ratio: {test_mdd_ratio:.4f}")
    print(f"True Label: {final_subject_true}")
    print(f"Predicted Label: {final_subject_pred}")

    test_segment_df = X_test_outer.copy()
    test_segment_df["channel"] = CHANNEL
    test_segment_df["model"] = "RF"
    test_segment_df["outer_fold"] = outer_fold
    test_segment_df["test_subject"] = test_subject
    test_segment_df["true_label"] = y_test_outer.values
    test_segment_df["segment_prediction"] = test_segment_preds
    test_segment_df["selected_threshold"] = best_threshold
    test_segment_df["test_mdd_ratio"] = test_mdd_ratio

    if "Segment" in df.columns:
        test_segment_df["Segment"] = df.iloc[test_idx]["Segment"].values

    test_segment_prediction_logs.append(test_segment_df)

    test_subject_logs.append({
        "channel": CHANNEL,
        "model": "RF",
        "outer_fold": outer_fold,
        "test_subject": test_subject,
        "true_label": final_subject_true,
        "predicted_label": final_subject_pred,
        "selected_threshold": best_threshold,
        "test_mdd_ratio": test_mdd_ratio,
        "num_segments": num_segments,
        "num_predicted_mdd_segments": num_mdd_segments,
        "num_predicted_control_segments": num_control_segments,
        "mdd_segments_over_total": f"{num_mdd_segments}/{num_segments}",
        "correct": int(final_subject_true == final_subject_pred)
    })

    fold_summary_logs.append({
        "channel": CHANNEL,
        "model": "RF",
        "outer_fold": outer_fold,
        "test_subject": test_subject,
        "best_inner_cv_accuracy": best_cv_score,
        "selected_threshold": best_threshold,
        "best_threshold_accuracy": best_threshold_score,
        "true_label": final_subject_true,
        "predicted_label": final_subject_pred,
        "test_mdd_ratio": test_mdd_ratio,
        "num_segments": num_segments,
        "num_predicted_mdd_segments": num_mdd_segments,
        "num_predicted_control_segments": num_control_segments,
        "mdd_segments_over_total": f"{num_mdd_segments}/{num_segments}",
        "correct": int(final_subject_true == final_subject_pred),
        **best_params
    })


# =====================================================
# Final results
# =====================================================

acc = accuracy_score(subject_true, subject_pred)

cm = confusion_matrix(subject_true, subject_pred)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

fold_correct = np.array(subject_true) == np.array(subject_pred)
outer_fold_accuracy_values = fold_correct.astype(float)

outer_fold_mean_accuracy = np.mean(outer_fold_accuracy_values)
outer_fold_std_accuracy = np.std(outer_fold_accuracy_values, ddof=1)


print("\n==============================")
print("OUTER-FOLD VARIABILITY")
print("==============================")
print(f"Outer-fold Accuracy Mean : {outer_fold_mean_accuracy:.3f}")
print(f"Outer-fold Accuracy SD   : {outer_fold_std_accuracy:.3f}")
print(f"Outer-fold Accuracy      : {outer_fold_mean_accuracy:.3f} ± {outer_fold_std_accuracy:.3f}")

print("\n==============================")
print("FINAL SUBJECT-LEVEL RESULTS")
print("==============================")
print(f"Accuracy    : {acc:.3f}")
print(f"Sensitivity : {sensitivity:.3f}")
print(f"Specificity : {specificity:.3f}")

print("\nConfusion Matrix:")
print(cm)

print("\nThresholds selected across folds:")
print(selected_thresholds)


# =====================================================
# Save outputs
# =====================================================

pd.DataFrame(fold_summary_logs).to_csv(
    os.path.join(output_dir, "fold_summary_logs.csv"),
    index=False
)

pd.concat(bayes_search_logs, ignore_index=True).to_csv(
    os.path.join(output_dir, "bayesian_search_all_iterations.csv"),
    index=False
)

pd.DataFrame(threshold_logs).to_csv(
    os.path.join(output_dir, "threshold_accuracy_logs.csv"),
    index=False
)

pd.DataFrame(inner_validation_logs).to_csv(
    os.path.join(output_dir, "inner_validation_subject_logs.csv"),
    index=False
)

pd.DataFrame(test_subject_logs).to_csv(
    os.path.join(output_dir, "test_subject_prediction_logs.csv"),
    index=False
)

pd.concat(test_segment_prediction_logs, ignore_index=True).to_csv(
    os.path.join(output_dir, "test_segment_prediction_logs.csv"),
    index=False
)

pd.DataFrame(
    cm,
    index=["True_Control", "True_MDD"],
    columns=["Predicted_Control", "Predicted_MDD"]
).to_csv(
    os.path.join(output_dir, "confusion_matrix.csv")
)

final_results = pd.DataFrame([{
    "channel": CHANNEL,
    "model": "RF",
    "accuracy": acc,
    "sensitivity": sensitivity,
    "specificity": specificity,
    "outer_fold_mean_accuracy": outer_fold_mean_accuracy,
    "outer_fold_std_accuracy": outer_fold_std_accuracy,
    "tn": tn,
    "fp": fp,
    "fn": fn,
    "tp": tp,
    "n_outer_folds": len(subject_true),
    "n_bayesian_iterations_per_fold": N_ITER,
    "random_state": RANDOM_STATE
}])

final_results.to_csv(
    os.path.join(output_dir, "final_subject_level_results.csv"),
    index=False
)

pd.DataFrame({
    "channel": CHANNEL,
    "model": "RF",
    "test_subject": [x["test_subject"] for x in test_subject_logs],
    "true_label": subject_true,
    "predicted_label": subject_pred,
    "selected_threshold": selected_thresholds,
    "test_mdd_ratio": [x["test_mdd_ratio"] for x in test_subject_logs],
    "num_predicted_mdd_segments": [x["num_predicted_mdd_segments"] for x in test_subject_logs],
    "num_segments": [x["num_segments"] for x in test_subject_logs],
    "mdd_segments_over_total": [x["mdd_segments_over_total"] for x in test_subject_logs],
    "correct": fold_correct.astype(int)
}).to_csv(
    os.path.join(output_dir, "final_true_pred_thresholds.csv"),
    index=False
)

pd.DataFrame({
    "channel": CHANNEL,
    "model": "RF",
    "outer_fold": list(range(1, len(selected_thresholds) + 1)),
    "test_subject": [x["test_subject"] for x in test_subject_logs],
    "selected_threshold": selected_thresholds
}).to_csv(
    os.path.join(output_dir, "selected_thresholds_across_folds.csv"),
    index=False
)

feature_rank_all = pd.concat(feature_rank_logs, ignore_index=True)

feature_rank_all.to_csv(
    os.path.join(output_dir, "feature_rank_each_fold_each_subject.csv"),
    index=False
)

feature_rank_summary = (
    feature_rank_all
    .groupby("feature")
    .agg(
        mean_importance=("importance", "mean"),
        std_importance=("importance", "std"),
        mean_rank=("rank", "mean"),
        median_rank=("rank", "median")
    )
    .reset_index()
    .sort_values(by=["mean_rank", "mean_importance"], ascending=[True, False])
)

feature_rank_summary["channel"] = CHANNEL
feature_rank_summary["model"] = "RF"

feature_rank_summary.to_csv(
    os.path.join(output_dir, "feature_rank_summary_across_folds.csv"),
    index=False
)

print("\nTop 20 features across all folds:")
print(feature_rank_summary.head(20))

print("\nAll logs saved to:")
print(output_dir)

FileNotFoundError: [Errno 2] No such file or directory: 'path_to_feature_files\\Feat_C4_final.csv'